# Model comparison

This notebook compares class-weighted logistic regression, weighted and unweighted XGBoost, and an unsupervised Isolation Forest on the chronological validation period. Average precision is the primary ranking metric because fraud is rare.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier

sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
DATA_PATH = next(
    (root / "data/raw/creditcard.csv" for root in (Path.cwd(), *Path.cwd().parents)
     if (root / "data/raw/creditcard.csv").exists()),
    None,
)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find data/raw/creditcard.csv in the current directory or its parents."
    )

df = pd.read_csv(DATA_PATH)
df = df.sort_values("Time").reset_index(drop=True)

train_end = int(len(df) * 0.60)
validation_end = int(len(df) * 0.80)

train_df = df.iloc[:train_end].copy()
validation_df = df.iloc[train_end:validation_end].copy()
test_df = df.iloc[validation_end:].copy()

FEATURES = [column for column in df.columns if column != "Class"]

X_train = train_df[FEATURES]
y_train = train_df["Class"]

X_validation = validation_df[FEATURES]
y_validation = validation_df["Class"]

X_test = test_df[FEATURES]
y_test = test_df["Class"]

print(f"Training fraud cases: {y_train.sum():,}")
print(f"Validation fraud cases: {y_validation.sum():,}")
print(f"Test fraud cases: {y_test.sum():,}")

Training fraud cases: 360
Validation fraud cases: 57
Test fraud cases: 75


In [2]:
scale_features = ["Time", "Amount"]
pca_features = [f"V{i}" for i in range(1, 29)]

preprocessor = ColumnTransformer(
    transformers=[
        ("scaled", RobustScaler(), scale_features),
        ("pca", "passthrough", pca_features),
    ]
)

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=2000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

logistic_model.fit(X_train, y_train)

logistic_scores = logistic_model.predict_proba(
    X_validation
)[:, 1]

In [3]:
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()
class_ratio = negative_count / positive_count

print(f"Legitimate training cases: {negative_count:,}")
print(f"Fraud training cases: {positive_count:,}")
print(f"Negative-to-positive ratio: {class_ratio:.2f}")

Legitimate training cases: 170,524
Fraud training cases: 360
Negative-to-positive ratio: 473.68


In [4]:
xgb_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    n_estimators=400,
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=class_ratio,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

xgb_model.fit(X_train, y_train)

xgb_scores = xgb_model.predict_proba(
    X_validation
)[:, 1]

In [5]:
xgb_unweighted = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    n_estimators=400,
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

xgb_unweighted.fit(X_train, y_train)

xgb_unweighted_scores = xgb_unweighted.predict_proba(
    X_validation
)[:, 1]

In [6]:
isolation_preprocessor = ColumnTransformer(
    transformers=[
        ("scaled", RobustScaler(), ["Time", "Amount"]),
        ("pca", "passthrough", pca_features),
    ]
)

X_train_transformed = isolation_preprocessor.fit_transform(X_train)
X_validation_transformed = isolation_preprocessor.transform(X_validation)

normal_train = X_train_transformed[y_train.to_numpy() == 0]

isolation_model = IsolationForest(
    n_estimators=300,
    max_samples="auto",
    contamination="auto",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

isolation_model.fit(normal_train)

,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",300
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for :meth:`fit`. ``None`` means 1unless in a :obj:`joblib.parallel_backend` context. ``-1`` means usingall processors. See :term:`Glossary <n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls the pseudo-randomness of the selection of the featureand split values for each branching step and each tree in the forest.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",'auto'
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0
,"bootstrap bootstrap: bool, default=FalseIf True, individual trees are fit on random subsets of the trainingdata sampled with replacement. If False, sampling without replacementis performed.",False
,"verbose verbose: int, default=0Controls the verbosity of the tree building process.",0
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fitand add more estimators to the ensemble, otherwise, just fit a wholenew forest. See :term:`the Glossary <warm_start>`... versionadded:: 0.21",False
Name,Type,Value
estimator_ estimator_: :class:`~sklearn.tree.ExtraTreeRegressor` instanceThe child estimator template used to create the collection offitted sub-estimators... versionadded:: 1.2 `base_estimator_` was renamed to `estimator_`.,ExtraTreeRegressor,ExtraTreeRegr...ndom_state=42)


In [7]:
isolation_scores = -isolation_model.score_samples(
    X_validation_transformed
)

## Validation comparison

Supervised models are evaluated with average precision and ROC-AUC. The Isolation Forest provides an unsupervised reference point trained only on legitimate transactions.

In [8]:
def evaluate_scores(y_true, scores, model_name):
    return {
        "model": model_name,
        "average_precision": average_precision_score(y_true, scores),
        "roc_auc": roc_auc_score(y_true, scores),
    }


results = pd.DataFrame(
    [
        evaluate_scores(
            y_validation,
            logistic_scores,
            "Class-weighted logistic regression",
        ),
        evaluate_scores(
            y_validation,
            xgb_unweighted_scores,
            "Unweighted XGBoost",
        ),
        evaluate_scores(
            y_validation,
            xgb_scores,
            "Class-weighted XGBoost",
        ),
        evaluate_scores(
            y_validation,
            isolation_scores,
            "Isolation Forest",
        ),
    ]
).sort_values("average_precision", ascending=False)

results

,model,average_precision,roc_auc
2,Class-weighted XGBoost,0.781009,0.977653
1,Unweighted XGBoost,0.775894,0.980989
0,Class-weighted logistic regression,0.772652,0.970903
3,Isolation Forest,0.019568,0.933829


In [9]:
def precision_recall_at_k(y_true, scores, k):
    y_array = np.asarray(y_true)
    scores_array = np.asarray(scores)

    ranked_indices = np.argsort(scores_array)[::-1]
    top_indices = ranked_indices[:k]

    fraud_detected = y_array[top_indices].sum()
    total_fraud = y_array.sum()

    return {
        "k": k,
        "fraud_detected": int(fraud_detected),
        "precision_at_k": fraud_detected / k,
        "recall_at_k": fraud_detected / total_fraud,
    }

## Interpretation

Model selection is based on both ranking quality and performance at a fixed review capacity. Small validation differences should not be treated as decisive without quantifying uncertainty; that analysis follows in notebook 05.